In [2]:
# ============================================================
# CLIENT DROPOUT EXPERIMENT
# Flat-FedAvg vs Hier-FedAvg
# 20 Clients - Strong Quantity Skew
#
# Dropout Scenarios:
#   10% -> 18 active / 2 dropped
#   20% -> 16 active / 4 dropped
#   40% -> 12 active / 8 dropped
#
# IMPORTANT:
# The original client datasets are NOT modified.
# Client dropout is simulated ONLY during FL training.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import os
import time
import random
import shutil

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. GOOGLE COLAB SETUP
# ============================================================

os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/fedavg_dropout_results", exist_ok=True)

print("=" * 80)
print("GOOGLE COLAB SETUP")
print("=" * 80)

print("Data directory    :", "/content/data")
print("Results directory :", "/content/fedavg_dropout_results")


# ============================================================
# 2. COPY RESULTS TO GOOGLE DRIVE
# ============================================================

drive_output_dir = "/content/drive/MyDrive/fedavg_dropout_results"

os.makedirs(
    drive_output_dir,
    exist_ok=True
)

print("\nGoogle Drive results directory:")
print(drive_output_dir)


# ============================================================
# 3. CONFIGURATION & HYPERPARAMETERS
# ============================================================

SEEDS = [42, 101]

NUM_CLIENTS = 20

GLOBAL_ROUNDS = 10

LOCAL_EPOCHS = 3

BATCH_SIZE = 256

LEARNING_RATE = 0.001

TEST_SIZE = 0.20


# ============================================================
# CLIENT DROPOUT SCENARIOS
# ============================================================

DROPOUT_RATES = [
    0.10,
    0.20,
    0.40
]


# ============================================================
# 4. HIERARCHICAL TOPOLOGY
# ============================================================

HFL_CLUSTERS = {

    'Station_1': [
        0, 1, 2, 3
    ],

    'Station_2': [
        4, 5, 6, 7
    ],

    'Station_3': [
        8, 9, 10, 11
    ],

    'Station_4': [
        12, 13, 14, 15
    ],

    'Station_5': [
        16, 17, 18, 19
    ]
}


# ============================================================
# 5. DATASET PATHS - GOOGLE DRIVE
# ============================================================

file_paths = [

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_1.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_2.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_3.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_4.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_5.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_6.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_7.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_8.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_9.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_10.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_11.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_12.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_13.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_14.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_15.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_16.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_17.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_18.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_19.csv',

    '/content/drive/MyDrive/Strong_Quantity_Skew/20_Clients/client_data_20.csv'
]


# ============================================================
# 6. OUTPUT DIRECTORY
# ============================================================

output_dir = "/content/fedavg_dropout_results"

os.makedirs(
    output_dir,
    exist_ok=True
)


# ============================================================
# 7. LABEL MAPPING
# ============================================================

label_mapping = {

    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 5

}


# ============================================================
# 8. LOAD CLIENT DATA
# ============================================================

def load_client_data(paths):

    client_dfs = []

    for client_id, path in enumerate(paths):

        if not os.path.exists(path):

            raise FileNotFoundError(
                f"File does not exist:\n{path}"
            )

        df = pd.read_csv(path)

        if df.empty:

            raise ValueError(
                f"Client {client_id + 1} dataset is empty."
            )

        if 'label' not in df.columns:

            raise ValueError(
                f"'label' column missing in "
                f"Client {client_id + 1}"
            )

        client_dfs.append(df)

    return client_dfs


client_dfs = load_client_data(
    file_paths
)


# ============================================================
# 9. VERIFY NUMBER OF CLIENTS
# ============================================================

if len(client_dfs) != NUM_CLIENTS:

    raise ValueError(
        f"Expected {NUM_CLIENTS} clients, "
        f"but found {len(client_dfs)}."
    )


# ============================================================
# 10. COMBINE DATA
# ============================================================

combined_df = pd.concat(
    client_dfs,
    ignore_index=True
)


combined_df['label'] = (
    combined_df['label']
    .map(label_mapping)
)


if combined_df['label'].isna().any():

    raise ValueError(
        "Unknown labels detected after label mapping."
    )


# ============================================================
# 11. GLOBAL FEATURES AND LABELS
# ============================================================

X_full_raw = (

    combined_df
    .drop('label', axis=1)
    .values
    .astype(np.float32)

)


y_full = (

    combined_df['label']
    .values
    .astype(np.int32)

)


# ============================================================
# 12. HANDLE NAN / INF
# ============================================================

X_full_raw = np.nan_to_num(

    X_full_raw,

    nan=0.0,

    posinf=0.0,

    neginf=0.0

)


# ============================================================
# 13. TRAIN / TEST SPLIT
# ============================================================

X_train_raw, X_test_raw, y_train, y_test = (

    train_test_split(

        X_full_raw,

        y_full,

        test_size=TEST_SIZE,

        random_state=42,

        stratify=y_full

    )

)


# ============================================================
# 14. STANDARDIZATION
# ============================================================

scaler = StandardScaler()


X_train = (

    scaler
    .fit_transform(X_train_raw)
    .astype(np.float32)

)


X_test = (

    scaler
    .transform(X_test_raw)
    .astype(np.float32)

)


# ============================================================
# 15. MODEL INPUT / CLASSES
# ============================================================

input_shape = (
    X_train.shape[1],
)


num_classes = len(
    np.unique(y_full)
)


# ============================================================
# 16. CREATE CLIENT DATA
# ============================================================

client_data_list = []


for client_id, df in enumerate(client_dfs):

    df = df.copy()

    df['label'] = (
        df['label']
        .map(label_mapping)
    )


    if df['label'].isna().any():

        raise ValueError(
            f"Unknown labels in "
            f"Client {client_id + 1}"
        )


    X_c_raw = (

        df
        .drop('label', axis=1)
        .values
        .astype(np.float32)

    )


    y_c = (

        df['label']
        .values
        .astype(np.int32)

    )


    X_c_raw = np.nan_to_num(

        X_c_raw,

        nan=0.0,

        posinf=0.0,

        neginf=0.0

    )


    X_c = (

        scaler
        .transform(X_c_raw)
        .astype(np.float32)

    )


    client_data_list.append(
        (X_c, y_c)
    )


# ============================================================
# 17. DATA INFORMATION
# ============================================================

print("\n" + "=" * 80)
print("DATASET INFORMATION")
print("=" * 80)

print(
    f"Total samples      : {len(X_full_raw)}"
)

print(
    f"Training samples   : {len(X_train)}"
)

print(
    f"Testing samples    : {len(X_test)}"
)

print(
    f"Number of features : {input_shape[0]}"
)

print(
    f"Number of classes  : {num_classes}"
)

print(
    f"Number of clients  : {NUM_CLIENTS}"
)


print("\nClient distribution:")

for client_idx, (X_c, y_c) in enumerate(
    client_data_list
):

    unique, counts = np.unique(
        y_c,
        return_counts=True
    )

    distribution = {

        int(k): int(v)

        for k, v in zip(
            unique,
            counts
        )

    }

    print(

        f"Client {client_idx + 1:02d}: "
        f"{len(X_c)} samples | "
        f"{distribution}"

    )


# ============================================================
# 18. TEST SET DISTRIBUTION
# ============================================================

print("\nTest-set distribution:")


unique_test, counts_test = np.unique(

    y_test,

    return_counts=True

)


for cls, count in zip(

    unique_test,

    counts_test

):

    print(
        f"Class {cls}: {count}"
    )


# ============================================================
# 19. MODEL ARCHITECTURE
# ============================================================

def create_mlp_model(
    input_shape,
    num_classes
):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=input_shape
        ),

        tf.keras.layers.Dense(
            128,
            activation='relu'
        ),

        tf.keras.layers.Dense(
            64,
            activation='relu'
        ),

        tf.keras.layers.Dense(
            32,
            activation='relu'
        ),

        tf.keras.layers.Dense(
            num_classes,
            activation='softmax'
        )

    ])

    return model


# ============================================================
# 20. MODEL SIZE
# ============================================================

def calculate_model_size_mb(model):

    total_params = model.count_params()

    return (

        total_params * 4

    ) / (

        1024 * 1024

    )


MODEL_SIZE_MB = calculate_model_size_mb(

    create_mlp_model(
        input_shape,
        num_classes
    )

)


print("\n" + "=" * 80)

print(
    f"MODEL SIZE: {MODEL_SIZE_MB:.4f} MB"
)

print("=" * 80)


# ============================================================
# 21. AGGREGATION
# ============================================================

def aggregate_weights(
    weights_list,
    sizes_list
):

    if len(weights_list) == 0:

        raise ValueError(
            "Cannot aggregate an empty list."
        )


    total_samples = sum(
        sizes_list
    )


    if total_samples <= 0:

        raise ValueError(
            "Total samples must be > 0."
        )


    aggregated_weights = []


    for layer_idx in range(
        len(weights_list[0])
    ):

        layer_avg = np.zeros_like(
            weights_list[0][layer_idx]
        )


        for client_idx in range(
            len(weights_list)
        ):

            factor = (

                sizes_list[client_idx]
                / total_samples

            )


            layer_avg += (

                factor
                * weights_list[
                    client_idx
                ][layer_idx]

            )


        aggregated_weights.append(
            layer_avg
        )


    return aggregated_weights


# ============================================================
# 22. FEDAVG CLIENT TRAINING
# ============================================================

def train_client_fedavg(
    global_weights,
    X_c,
    y_c
):

    model = create_mlp_model(

        input_shape,

        num_classes

    )


    model.set_weights(
        global_weights
    )


    model.compile(

        optimizer=tf.keras.optimizers.Adam(

            learning_rate=LEARNING_RATE

        ),

        loss='sparse_categorical_crossentropy'

    )


    model.fit(

        X_c,

        y_c,

        epochs=LOCAL_EPOCHS,

        batch_size=BATCH_SIZE,

        verbose=0

    )


    weights = model.get_weights()


    tf.keras.backend.clear_session()


    return weights


# ============================================================
# 23. EVALUATION
# ============================================================

def evaluate_model(model):

    probabilities = model.predict(

        X_test,

        batch_size=BATCH_SIZE,

        verbose=0

    )


    predictions = np.argmax(

        probabilities,

        axis=1

    )


    accuracy = accuracy_score(

        y_test,

        predictions

    )


    macro_precision = precision_score(

        y_test,

        predictions,

        average='macro',

        zero_division=0

    )


    macro_recall = recall_score(

        y_test,

        predictions,

        average='macro',

        zero_division=0

    )


    macro_f1 = f1_score(

        y_test,

        predictions,

        average='macro',

        zero_division=0

    )


    class_report = classification_report(

        y_test,

        predictions,

        labels=np.arange(num_classes),

        output_dict=True,

        zero_division=0

    )


    cm = confusion_matrix(

        y_test,

        predictions,

        labels=np.arange(num_classes)

    )


    row_sums = cm.sum(

        axis=1,

        keepdims=True

    )


    cm_norm = np.divide(

        cm.astype(np.float64),

        row_sums,

        out=np.zeros_like(

            cm,

            dtype=np.float64

        ),

        where=row_sums != 0

    )


    return {

        'accuracy':
            accuracy,

        'macro_precision':
            macro_precision,

        'macro_recall':
            macro_recall,

        'macro_f1':
            macro_f1,

        'class_report':
            class_report,

        'cm_norm':
            cm_norm

    }


# ============================================================
# 24. DROPOUT CALCULATION
# ============================================================

def calculate_dropout_clients(
    dropout_rate
):

    num_dropped = int(

        round(
            NUM_CLIENTS * dropout_rate
        )

    )


    num_active = (

        NUM_CLIENTS
        - num_dropped

    )


    if num_active <= 0:

        raise ValueError(
            "Dropout rate leaves "
            "zero active clients."
        )


    return num_active, num_dropped


# ============================================================
# 25. SELECT DROPPED CLIENTS
# ============================================================

def select_dropped_clients(
    dropout_rate,
    seed,
    round_idx
):

    num_active, num_dropped = (

        calculate_dropout_clients(
            dropout_rate
        )

    )


    if num_dropped == 0:

        return [], list(
            range(NUM_CLIENTS)
        )


    # --------------------------------------------------------
    # Deterministic selection for reproducibility
    # --------------------------------------------------------

    round_rng = random.Random(

        seed
        + (round_idx + 1) * 10000

        + int(
            dropout_rate * 1000
        )

    )


    dropped_clients = sorted(

        round_rng.sample(

            range(NUM_CLIENTS),

            num_dropped

        )

    )


    dropped_set = set(
        dropped_clients
    )


    active_clients = [

        idx

        for idx in range(NUM_CLIENTS)

        if idx not in dropped_set

    ]


    return (

        dropped_clients,

        active_clients

    )


# ============================================================
# 26. FL EXPERIMENT WITH CLIENT DROPOUT
# ============================================================

def run_fl_experiment(

    method,

    seed,

    dropout_rate

):

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    np.random.seed(seed)

    random.seed(seed)

    tf.random.set_seed(seed)


    # --------------------------------------------------------
    # Global model
    # --------------------------------------------------------

    global_model = create_mlp_model(

        input_shape,

        num_classes

    )


    total_comm_mb = 0.0


    experiment_start = time.time()


    round_accuracy = []

    round_f1 = []

    round_precision = []

    round_recall = []

    round_time = []


    dropout_details = []


    # ========================================================
    # GLOBAL ROUNDS
    # ========================================================

    for round_idx in range(
        GLOBAL_ROUNDS
    ):

        round_start = time.time()


        global_weights = (

            global_model.get_weights()

        )


        # ----------------------------------------------------
        # Select dropped clients
        # ----------------------------------------------------

        (

            dropped_clients,

            active_clients

        ) = select_dropped_clients(

            dropout_rate,

            seed,

            round_idx

        )


        dropped_client_names = [

            f"Client {idx + 1}"

            for idx in dropped_clients

        ]


        active_client_names = [

            f"Client {idx + 1}"

            for idx in active_clients

        ]


        print(

            f"\n      Round {round_idx + 1:02d} | "

            f"Active: {len(active_clients)} | "

            f"Dropped: {len(dropped_clients)}"

        )


        print(

            f"      Dropped clients: "

            f"{dropped_client_names}"

        )


        # ====================================================
        # FLAT FEDAVG
        # ====================================================

        if method == 'Flat-FedAvg':

            local_weights = []

            local_sizes = []


            for client_idx in active_clients:

                X_c, y_c = (

                    client_data_list[
                        client_idx
                    ]

                )


                client_weights = (

                    train_client_fedavg(

                        global_weights,

                        X_c,

                        y_c

                    )

                )


                local_weights.append(

                    client_weights

                )


                local_sizes.append(

                    len(X_c)

                )


                # --------------------------------------------
                # Client -> Global
                # Global -> Client
                # --------------------------------------------

                total_comm_mb += (

                    MODEL_SIZE_MB * 2

                )


            # ------------------------------------------------
            # Global aggregation
            # ------------------------------------------------

            if len(local_weights) == 0:

                raise RuntimeError(
                    "No active clients."
                )


            new_global_weights = (

                aggregate_weights(

                    local_weights,

                    local_sizes

                )

            )


            global_model.set_weights(

                new_global_weights

            )


        # ====================================================
        # HIERARCHICAL FEDAVG
        # ====================================================

        elif method == 'Hier-FedAvg':

            station_weights = []

            station_sizes = []


            for station, client_indices in (

                HFL_CLUSTERS.items()

            ):

                station_local_weights = []

                station_local_sizes = []


                # --------------------------------------------
                # Only active clients in this station
                # --------------------------------------------

                station_active_clients = [

                    client_idx

                    for client_idx in client_indices

                    if client_idx in active_clients

                ]


                station_dropped_clients = [

                    client_idx

                    for client_idx in client_indices

                    if client_idx in dropped_clients

                ]


                # --------------------------------------------
                # Train active clients
                # --------------------------------------------

                for client_idx in (

                    station_active_clients

                ):

                    X_c, y_c = (

                        client_data_list[
                            client_idx
                        ]

                    )


                    client_weights = (

                        train_client_fedavg(

                            global_weights,

                            X_c,

                            y_c

                        )

                    )


                    station_local_weights.append(

                        client_weights

                    )


                    station_local_sizes.append(

                        len(X_c)

                    )


                    # ----------------------------------------
                    # Client <-> Station
                    # ----------------------------------------

                    total_comm_mb += (

                        MODEL_SIZE_MB * 2

                    )


                # --------------------------------------------
                # Station has no active clients
                # --------------------------------------------

                if len(
                    station_local_weights
                ) == 0:

                    print(

                        f"        {station}: "

                        f"NO ACTIVE CLIENTS "

                        f"-> skipped"

                    )

                    continue


                # --------------------------------------------
                # Station aggregation
                # --------------------------------------------

                station_agg_weights = (

                    aggregate_weights(

                        station_local_weights,

                        station_local_sizes

                    )

                )


                station_weights.append(

                    station_agg_weights

                )


                station_sizes.append(

                    sum(
                        station_local_sizes
                    )

                )


                print(

                    f"        {station}: "

                    f"{len(station_active_clients)} "
                    f"active, "

                    f"{len(station_dropped_clients)} "
                    f"dropped"

                )


                # --------------------------------------------
                # Station <-> Global Server
                # --------------------------------------------

                total_comm_mb += (

                    MODEL_SIZE_MB * 2

                )


            # ------------------------------------------------
            # Global aggregation
            # ------------------------------------------------

            if len(station_weights) == 0:

                raise RuntimeError(

                    "No active stations "
                    "available for aggregation."

                )


            new_global_weights = (

                aggregate_weights(

                    station_weights,

                    station_sizes

                )

            )


            global_model.set_weights(

                new_global_weights

            )


        else:

            raise ValueError(

                f"Unknown FL method: {method}"

            )


        # ====================================================
        # ROUND EVALUATION
        # ====================================================

        round_result = evaluate_model(

            global_model

        )


        round_accuracy.append(

            round_result[
                'accuracy'
            ]

        )


        round_f1.append(

            round_result[
                'macro_f1'
            ]

        )


        round_precision.append(

            round_result[
                'macro_precision'
            ]

        )


        round_recall.append(

            round_result[
                'macro_recall'
            ]

        )


        elapsed_round = (

            time.time()
            - round_start

        )


        round_time.append(

            elapsed_round

        )


        # ----------------------------------------------------
        # Save dropout information
        # ----------------------------------------------------

        dropout_details.append({

            'Method':
                method,

            'Seed':
                seed,

            'Dropout_Rate':
                dropout_rate,

            'Round':
                round_idx + 1,

            'Total_Clients':
                NUM_CLIENTS,

            'Active_Clients':
                len(active_clients),

            'Dropped_Clients':
                len(dropped_clients),

            'Active_Client_IDs':
                ','.join(
                    str(i + 1)
                    for i in active_clients
                ),

            'Dropped_Client_IDs':
                ','.join(
                    str(i + 1)
                    for i in dropped_clients
                ),

            'Accuracy':
                round_result[
                    'accuracy'
                ],

            'Macro_F1':
                round_result[
                    'macro_f1'
                ],

            'Macro_Precision':
                round_result[
                    'macro_precision'
                ],

            'Macro_Recall':
                round_result[
                    'macro_recall'
                ],

            'Round_Time_Sec':
                elapsed_round

        })


        print(

            f"      Accuracy: "
            f"{round_result['accuracy']:.4f} | "

            f"Macro-F1: "
            f"{round_result['macro_f1']:.4f}"

        )


    # ========================================================
    # FINAL EVALUATION
    # ========================================================

    final_result = evaluate_model(

        global_model

    )


    total_time = (

        time.time()
        - experiment_start

    )


    return {

        'accuracy':
            final_result[
                'accuracy'
            ],

        'macro_precision':
            final_result[
                'macro_precision'
            ],

        'macro_recall':
            final_result[
                'macro_recall'
            ],

        'macro_f1':
            final_result[
                'macro_f1'
            ],

        'comm_mb':
            total_comm_mb,

        'time_sec':
            total_time,

        'class_report':
            final_result[
                'class_report'
            ],

        'cm_norm':
            final_result[
                'cm_norm'
            ],

        'round_accuracy':
            round_accuracy,

        'round_f1':
            round_f1,

        'round_precision':
            round_precision,

        'round_recall':
            round_recall,

        'round_time':
            round_time,

        'dropout_details':
            dropout_details

    }


# ============================================================
# 27. METHODS
# ============================================================

methods = [

    'Flat-FedAvg',

    'Hier-FedAvg'

]


# ============================================================
# 28. STORAGE
# ============================================================

summary_results = []

convergence_results = []

dropout_round_results = []


detailed_class_metrics = {

    method: {

        dropout_rate: []

        for dropout_rate
        in DROPOUT_RATES

    }

    for method in methods

}


# ============================================================
# 29. START BENCHMARK
# ============================================================

print("\n" + "=" * 80)

print(
    "CLIENT DROPOUT FEDAVG EXPERIMENT"
)

print("=" * 80)

print(
    "20 Clients - Strong Quantity Skew"
)

print(
    "Flat-FedAvg vs Hier-FedAvg"
)

print(
    "Seeds:",
    SEEDS
)

print(
    "Dropout scenarios:",
    DROPOUT_RATES
)

print("=" * 80)


# ============================================================
# 30. RUN ALL DROPOUT SCENARIOS
# ============================================================

for dropout_rate in DROPOUT_RATES:

    num_active, num_dropped = (

        calculate_dropout_clients(
            dropout_rate
        )

    )


    print("\n" + "#" * 80)

    print(

        f"DROPOUT SCENARIO: "
        f"{dropout_rate * 100:.0f}%"

    )

    print(

        f"Active Clients : "
        f"{num_active}"

    )

    print(

        f"Dropped Clients: "
        f"{num_dropped}"

    )

    print("#" * 80)


    for method in methods:

        print("\n" + "-" * 80)

        print(
            f"METHOD: {method}"
        )

        print("-" * 80)


        accuracy_values = []

        f1_values = []

        precision_values = []

        recall_values = []

        communication_values = []

        time_values = []


        for seed in SEEDS:

            print(
                f"\n  Seed {seed}"
            )


            result = run_fl_experiment(

                method,

                seed,

                dropout_rate

            )


            accuracy_values.append(

                result[
                    'accuracy'
                ]

            )


            f1_values.append(

                result[
                    'macro_f1'
                ]

            )


            precision_values.append(

                result[
                    'macro_precision'
                ]

            )


            recall_values.append(

                result[
                    'macro_recall'
                ]

            )


            communication_values.append(

                result[
                    'comm_mb'
                ]

            )


            time_values.append(

                result[
                    'time_sec'
                ]

            )


            detailed_class_metrics[
                method
            ][
                dropout_rate
            ].append(

                result

            )


            # ------------------------------------------------
            # Convergence
            # ------------------------------------------------

            for round_idx in range(

                GLOBAL_ROUNDS

            ):

                convergence_results.append({

                    'Method':
                        method,

                    'Dropout_Rate':
                        dropout_rate,

                    'Dropout_Percentage':
                        dropout_rate * 100,

                    'Seed':
                        seed,

                    'Round':
                        round_idx + 1,

                    'Active_Clients':
                        NUM_CLIENTS
                        - int(
                            NUM_CLIENTS
                            * dropout_rate
                        ),

                    'Accuracy':
                        result[
                            'round_accuracy'
                        ][round_idx],

                    'Macro_F1':
                        result[
                            'round_f1'
                        ][round_idx],

                    'Macro_Precision':
                        result[
                            'round_precision'
                        ][round_idx],

                    'Macro_Recall':
                        result[
                            'round_recall'
                        ][round_idx],

                    'Round_Time_Sec':
                        result[
                            'round_time'
                        ][round_idx]

                })


            # ------------------------------------------------
            # Dropout round details
            # ------------------------------------------------

            dropout_round_results.extend(

                result[
                    'dropout_details'
                ]

            )


            print(

                f"  Final Accuracy: "
                f"{result['accuracy']:.4f}"

            )


            print(

                f"  Final Macro-F1: "
                f"{result['macro_f1']:.4f}"

            )


            print(

                f"  Communication: "
                f"{result['comm_mb']:.2f} MB"

            )


            print(

                f"  Time: "
                f"{result['time_sec']:.2f} sec"

            )


        # ====================================================
        # SUMMARY
        # ====================================================

        summary_results.append({

            'Method':
                method,

            'Dropout_Rate':
                dropout_rate,

            'Dropout_Percentage':
                dropout_rate * 100,

            'Total_Clients':
                NUM_CLIENTS,

            'Active_Clients':
                num_active,

            'Dropped_Clients':
                num_dropped,

            'Accuracy_Mean':
                np.mean(
                    accuracy_values
                ),

            'Accuracy_Std':
                np.std(
                    accuracy_values
                ),

            'Macro_F1_Mean':
                np.mean(
                    f1_values
                ),

            'Macro_F1_Std':
                np.std(
                    f1_values
                ),

            'Macro_Precision_Mean':
                np.mean(
                    precision_values
                ),

            'Macro_Precision_Std':
                np.std(
                    precision_values
                ),

            'Macro_Recall_Mean':
                np.mean(
                    recall_values
                ),

            'Macro_Recall_Std':
                np.std(
                    recall_values
                ),

            'Comm_Cost_MB_Mean':
                np.mean(
                    communication_values
                ),

            'Comm_Cost_MB_Std':
                np.std(
                    communication_values
                ),

            'Exec_Time_Sec_Mean':
                np.mean(
                    time_values
                ),

            'Exec_Time_Sec_Std':
                np.std(
                    time_values
                )

        })


# ============================================================
# 31. OVERALL SUMMARY
# ============================================================

df_summary = pd.DataFrame(

    summary_results

)


summary_path = os.path.join(

    output_dir,

    'fedavg_client_dropout_summary.csv'

)


df_summary.to_csv(

    summary_path,

    index=False

)


# ============================================================
# PRINT SUMMARY
# ============================================================

print("\n" + "=" * 80)

print(
    "CLIENT DROPOUT SUMMARY"
)

print("=" * 80)


print(

    df_summary.round(4).to_string(
        index=False
    )

)


# ============================================================
# 32. CONVERGENCE CSV
# ============================================================

df_convergence = pd.DataFrame(

    convergence_results

)


convergence_path = os.path.join(

    output_dir,

    'fedavg_client_dropout_convergence.csv'

)


df_convergence.to_csv(

    convergence_path,

    index=False

)


# ============================================================
# 33. DROPOUT ROUND DETAILS CSV
# ============================================================

df_dropout_rounds = pd.DataFrame(

    dropout_round_results

)


dropout_round_path = os.path.join(

    output_dir,

    'fedavg_client_dropout_round_details.csv'

)


df_dropout_rounds.to_csv(

    dropout_round_path,

    index=False

)


# ============================================================
# 34. CLASS-WISE PERFORMANCE
# ============================================================

print("\n" + "=" * 80)

print(
    "CLIENT DROPOUT CLASS-WISE PERFORMANCE"
)

print("=" * 80)


for method in methods:

    for dropout_rate in DROPOUT_RATES:

        reports = [

            run[
                'class_report'
            ]

            for run in
            detailed_class_metrics[
                method
            ][
                dropout_rate
            ]

        ]


        class_rows = []


        for c in range(
            num_classes
        ):

            c_str = str(c)


            precision_values = [

                report[
                    c_str
                ]['precision']

                for report in reports

            ]


            recall_values = [

                report[
                    c_str
                ]['recall']

                for report in reports

            ]


            f1_values_class = [

                report[
                    c_str
                ]['f1-score']

                for report in reports

            ]


            support_values = [

                report[
                    c_str
                ]['support']

                for report in reports

            ]


            class_rows.append({

                'Class':
                    f'Class {c}',

                'Support':
                    np.mean(
                        support_values
                    ),

                'Precision_Mean':
                    np.mean(
                        precision_values
                    ),

                'Precision_Std':
                    np.std(
                        precision_values
                    ),

                'Recall_Mean':
                    np.mean(
                        recall_values
                    ),

                'Recall_Std':
                    np.std(
                        recall_values
                    ),

                'F1_Mean':
                    np.mean(
                        f1_values_class
                    ),

                'F1_Std':
                    np.std(
                        f1_values_class
                    )

            })


        df_class = pd.DataFrame(

            class_rows

        )


        dropout_label = (

            f"{dropout_rate * 100:.0f}"

        )


        filename = (

            method
            .lower()
            .replace('-', '_')

            + f'_dropout_{dropout_label}pct_'

            + 'class_wise_performance.csv'

        )


        class_path = os.path.join(

            output_dir,

            filename

        )


        df_class.to_csv(

            class_path,

            index=False

        )


        print(

            f"\n--- {method} | "
            f"{dropout_label}% Dropout ---"

        )


        print(

            df_class.round(4).to_string(
                index=False
            )

        )


# ============================================================
# 35. CONFUSION MATRICES
# ============================================================

print("\n" + "=" * 80)

print(
    "AVERAGE NORMALIZED CONFUSION MATRICES"
)

print("=" * 80)


for method in methods:

    for dropout_rate in DROPOUT_RATES:

        print("\n" + "-" * 80)

        print(

            f"{method} | "
            f"Dropout = "
            f"{dropout_rate * 100:.0f}%"

        )

        print("-" * 80)


        matrices = [

            run[
                'cm_norm'
            ]

            for run in
            detailed_class_metrics[
                method
            ][
                dropout_rate
            ]

        ]


        avg_cm = np.mean(

            matrices,

            axis=0

        )


        df_cm = pd.DataFrame(

            avg_cm,

            index=[

                f'Class {i}'

                for i in range(
                    num_classes
                )

            ],

            columns=[

                f'Class {i}'

                for i in range(
                    num_classes
                )

            ]

        )


        filename_base = (

            method
            .lower()
            .replace('-', '_')

            + f'_dropout_'
            + f'{dropout_rate * 100:.0f}pct'

        )


        # ====================================================
        # SAVE CSV
        # ====================================================

        csv_path = os.path.join(

            output_dir,

            filename_base
            + '_normalized_confusion_matrix.csv'

        )


        df_cm.to_csv(

            csv_path

        )


        print(

            "\nAverage Normalized Confusion Matrix:"

        )


        print(

            df_cm.round(4).to_string()

        )


        # ====================================================
        # CREATE PNG
        # ====================================================

        fig, ax = plt.subplots(

            figsize=(9, 8)

        )


        im = ax.imshow(

            avg_cm,

            interpolation='nearest'

        )


        cbar = ax.figure.colorbar(

            im,

            ax=ax

        )


        cbar.ax.set_ylabel(

            'Normalized Value',

            rotation=-90,

            va='bottom'

        )


        ax.set(

            xticks=np.arange(
                num_classes
            ),

            yticks=np.arange(
                num_classes
            ),

            xticklabels=[

                f'Class {i}'

                for i in range(
                    num_classes
                )

            ],

            yticklabels=[

                f'Class {i}'

                for i in range(
                    num_classes
                )

            ],

            ylabel='True Label',

            xlabel='Predicted Label',

            title=(

                f'{method}\n'

                f'{dropout_rate * 100:.0f}% '
                f'Client Dropout\n'

                'Average Normalized '
                'Confusion Matrix'

            )

        )


        plt.setp(

            ax.get_xticklabels(),

            rotation=45,

            ha='right',

            rotation_mode='anchor'

        )


        threshold = (

            avg_cm.max()
            / 2.0

        )


        for i in range(
            num_classes
        ):

            for j in range(
                num_classes
            ):

                value = avg_cm[i, j]


                ax.text(

                    j,

                    i,

                    f'{value:.2f}',

                    ha='center',

                    va='center',

                    color=(

                        'white'

                        if value > threshold

                        else 'black'

                    ),

                    fontsize=11

                )


        fig.tight_layout()


        image_path = os.path.join(

            output_dir,

            filename_base
            + '_normalized_confusion_matrix.png'

        )


        plt.savefig(

            image_path,

            dpi=300,

            bbox_inches='tight'

        )


        plt.close(fig)


        print("\nCSV saved:")

        print(csv_path)


        print("\nPNG saved:")

        print(image_path)


# ============================================================
# 36. SAVE DROPOUT SCENARIO SUMMARY SEPARATELY
# ============================================================

scenario_summary = (

    df_summary[

        [

            'Method',

            'Dropout_Percentage',

            'Active_Clients',

            'Dropped_Clients',

            'Accuracy_Mean',

            'Accuracy_Std',

            'Macro_F1_Mean',

            'Macro_F1_Std',

            'Macro_Precision_Mean',

            'Macro_Precision_Std',

            'Macro_Recall_Mean',

            'Macro_Recall_Std',

            'Comm_Cost_MB_Mean',

            'Exec_Time_Sec_Mean'

        ]

    ]

)


scenario_path = os.path.join(

    output_dir,

    'client_dropout_scenario_comparison.csv'

)


scenario_summary.to_csv(

    scenario_path,

    index=False

)


# ============================================================
# 37. COPY ALL RESULTS TO GOOGLE DRIVE
# ============================================================

print("\n" + "=" * 80)

print(
    "COPYING RESULTS TO GOOGLE DRIVE"
)

print("=" * 80)


shutil.copytree(

    output_dir,

    drive_output_dir,

    dirs_exist_ok=True

)


print(

    "Results copied to:"

)

print(

    drive_output_dir

)


# ============================================================
# 38. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 80)

print(
    "[FINISHED] CLIENT DROPOUT EXPERIMENT COMPLETED"
)

print("=" * 80)


print("\nMethods evaluated:")

for method in methods:

    print(
        f"  ✓ {method}"
    )


print("\nDropout scenarios:")

for dropout_rate in DROPOUT_RATES:

    active, dropped = (

        calculate_dropout_clients(
            dropout_rate
        )

    )

    print(

        f"  ✓ "
        f"{dropout_rate * 100:.0f}% dropout "
        f"-> "
        f"{active} active / "
        f"{dropped} dropped"

    )


print("\nMetrics generated:")

print("  ✓ Accuracy")
print("  ✓ Macro Precision")
print("  ✓ Macro Recall")
print("  ✓ Macro F1")
print("  ✓ Class-wise Precision")
print("  ✓ Class-wise Recall")
print("  ✓ Class-wise F1")
print("  ✓ Class Support")
print("  ✓ Normalized Confusion Matrices")
print("  ✓ Confusion Matrix PNG Images")
print("  ✓ Round-by-round Convergence")
print("  ✓ Client Dropout Per Round")
print("  ✓ Active Client IDs")
print("  ✓ Dropped Client IDs")
print("  ✓ Communication Cost")
print("  ✓ Execution Time")
print("  ✓ Mean ± Std across seeds")


print("\nMain summary:")

print(summary_path)


print("\nConvergence:")

print(convergence_path)


print("\nDropout details:")

print(dropout_round_path)


print("\nScenario comparison:")

print(scenario_path)


print("\nAll local results:")

print(output_dir)


print("\nGoogle Drive results:")

print(drive_output_dir)


print("\n" + "=" * 80)

GOOGLE COLAB SETUP
Data directory    : /content/data
Results directory : /content/fedavg_dropout_results

Google Drive results directory:
/content/drive/MyDrive/fedavg_dropout_results

DATASET INFORMATION
Total samples      : 1853190
Training samples   : 1482552
Testing samples    : 370638
Number of features : 23
Number of classes  : 6
Number of clients  : 20

Client distribution:
Client 01: 159414 samples | {0: 112136, 1: 22401, 2: 6220, 3: 6219, 4: 6219, 5: 6219}
Client 02: 151443 samples | {0: 106529, 1: 21281, 2: 5909, 3: 5908, 4: 5908, 5: 5908}
Client 03: 143473 samples | {0: 100923, 1: 20161, 2: 5598, 3: 5597, 4: 5597, 5: 5597}
Client 04: 135502 samples | {0: 95316, 1: 19041, 2: 5287, 3: 5286, 4: 5286, 5: 5286}
Client 05: 127531 samples | {0: 89709, 1: 17921, 2: 4976, 3: 4975, 4: 4975, 5: 4975}
Client 06: 119561 samples | {0: 84103, 1: 16801, 2: 4665, 3: 4664, 4: 4664, 5: 4664}
Client 07: 111590 samples | {0: 78496, 1: 15681, 2: 4354, 3: 4353, 4: 4353, 5: 4353}
Client 08: 103619 

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
